# Faruq-v3 — CAFR C1 seed42
Satu runtime Colab khusus **C1**. Boleh dijalankan paralel dengan arm lain.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess, sys, tarfile
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
REPO=Path('/content/coffee-bean-detection')
BRANCH='agent/cafr-yolo'
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); os.chdir(REPO)

from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
))
ARCHIVE=require_project_artifact(PROJECT_ROOT,'bundles/faruq-development-v3-grouped.tar')
D0=require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
DATA=Path('/content/faruq-development-v3-grouped')
if DATA.exists(): shutil.rmtree(DATA)
with tarfile.open(ARCHIVE,'r') as a: a.extractall('/content',filter='data')
assert not (DATA/'test').exists(),'STOP: split test tidak boleh tersedia.'
OUTPUT=PROJECT_ROOT/'experiments/faruq-v3-cafr-seed42-v1'
OUTPUT.mkdir(parents=True,exist_ok=True)

ARM='C1'
cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_cafr_arm',
     '--arm',ARM,'--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),
     '--d0-checkpoint',str(D0),'--output-root',str(OUTPUT),'--seed','42','--device','0',
     '--latency-iterations','50','--authorize-training']
print('GPU:',torch.cuda.get_device_name(0)); print('RUN:',ARM); print('OUTPUT:',OUTPUT)
subprocess.run(cmd,cwd=REPO,check=True)

import json,pandas as pd
from IPython.display import display
result=json.loads((OUTPUT/'val_reports'/f'{ARM}_seed42_result.json').read_text())
m=result['metrics']
display(pd.DataFrame([{'arm':ARM,'macro_mAP50_95':m.get('macro_map50_95'),
'bottom3_mAP50_95':m.get('bottom3_class_map50_95'),'worst_mAP50_95':m.get('worst_class_map50_95'),
'mAP50_95':m.get('metrics/mAP50-95(B)'),'latency_ms':result.get('latency',{}).get('median_ms'),
'patch_size':result.get('cafr',{}).get('patch_size')}]))
print('Kirim tabel ini ke chat.')
